# Paper Figures

**SQ1** Model comparison (CV5): bar charts, hydrographs, scatter, FDC, peak analysis

**SQ2** Extreme holdout: standard vs extreme, HMS+LSTM blend, HMS calibration comparison

**SQ3** BC/Flow ablation: delta charts, IG attribution, attribution vs necessity

**Discussion** IG heatmaps, seasonal decomposition, peak close-ups

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from xarray import DataArray
from matplotlib.patches import Patch

from UCB_training.UCB_plotting import plot_timeseries_comparison, plot_forecasts, plot_lstm_vs_pilstm
from UCB_training.UCB_utils import data_dir, clean_df
from neuralhydrology.evaluation.metrics import calculate_all_metrics

# Bumped font sizes ~20% for journal print
plt.rcParams.update({'font.size': 14, 'axes.titlesize': 15, 'axes.labelsize': 13, 'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 10, 'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight'})

PROJECT = Path(os.getcwd()).parent.parent
OUTPUTS = PROJECT / 'outputs'
DATA_DIR = PROJECT / 'russian_river_data'
FIG_DIR = PROJECT / 'outputs' / '_all_basins' / 'paper_figs'
FIG_DIR.mkdir(parents=True, exist_ok=True)
SAVE_DIR = FIG_DIR

BASINS = ['guerneville', 'hopland', 'calpella', 'warm_springs']
BASIN_LABELS = {'guerneville': 'Guerneville', 'hopland': 'Hopland', 'calpella': 'Calpella', 'warm_springs': 'Warm Springs'}
NOBC_BASINS = ['guerneville', 'hopland', 'calpella']

# Global color scheme - blue/amber/green, red reserved for blend
COLORS = {'HMS': '#1E88E5', 'LSTM': '#F9A825', 'PILSTM': '#43A047'}
MODEL_STYLES = {
    'HMS':    {'color': '#1E88E5', 'ls': '--', 'lw': 1.5},
    'LSTM':   {'color': '#F9A825', 'ls': '-',  'lw': 1.5},
    'PILSTM': {'color': '#43A047', 'ls': '-',  'lw': 1.5},
}
MODELS = ['HMS', 'LSTM', 'PILSTM']

print('Project root:', PROJECT)
print('Fig output:', FIG_DIR)

In [ ]:
HMS_FLOW_COL = {'guerneville': 'Guerneville Gage FLOW', 'hopland': 'Hopland Gage FLOW', 'calpella': 'Capella Gage FLOW', 'warm_springs': 'Warm Springs Dam Inflow FLOW'}
HMS_FILES = {
    'guerneville': {'1D': 'Guerneville_daily_averaged.csv', '1H': 'Guerneville_hourly.csv'},
    'hopland': {'1D': 'Hopland_daily_averaged.csv', '1H': 'Hopland_hourly.csv'},
    'calpella': {'1D': 'Calpella_daily_averaged.csv', '1H': 'Calpella_hourly.csv'},
    'warm_springs': {'1D': 'WarmSprings_Inflow_daily_averaged.csv', '1H': 'WarmSprings_Inflow_hourly.csv'},
}
HMS_FILES_EXTREME = {b: {s: f.replace('.csv', '_extreme.csv') for s, f in files.items()} for b, files in HMS_FILES.items()}

def find_latest_run(basin, exp_label):
    runs_dir = OUTPUTS / basin / 'mts_shared' / 'runs'
    matches = sorted([d for d in runs_dir.iterdir() if d.is_dir() and d.name.startswith(f'{exp_label}_') and 'grid' not in d.name], key=lambda x: x.name, reverse=True)
    return matches[0] if matches else None

def load_ensemble(run_dir, model_type, scale):
    prefix = 'nophys' if model_type == 'lstm' else 'phys'
    p = run_dir / f'{prefix}_member_run_01' / f'{prefix}_ensemble' / f'results_output_test_{scale}.csv'
    if not p.exists(): return None
    df = pd.read_csv(p, parse_dates=['Date'])
    df['Predicted'] = df['Predicted'].clip(lower=0)  # clip negative predictions
    return df

def load_hms(basin, scale, extreme=False):
    files = HMS_FILES_EXTREME if extreme else HMS_FILES
    p = DATA_DIR / files[basin][scale]
    if not p.exists():
        print(f'  WARNING: {p.name} not found')
        return None
    raw = pd.read_csv(p, low_memory=False)
    df = clean_df(raw)
    col = HMS_FLOW_COL[basin]
    if col not in df.columns:
        print(f'  WARNING: {col} not in {p.name}')
        return None
    out = df[[col]].rename(columns={col: 'HMS'}).copy()
    out['HMS'] = out['HMS'].clip(lower=0)  # clip negative HMS
    out.index.name = 'Date'
    if scale == '1D':
        out = out.resample('1D').mean()
    return out.reset_index()

EXPERIMENT_LABELS = {'CV5': 'CROSS_VAL_V5', 'NOBC': 'MTS_NOBC_CV5', 'NOBC_V2': 'MTS_NOBC_V2_CV5', 'EXTREME': 'EXTREME_SEQ_A_CV_V2'}
EXPERIMENT_BASINS = {'CV5': BASINS, 'NOBC': NOBC_BASINS, 'NOBC_V2': NOBC_BASINS, 'EXTREME': BASINS}

data = {}
run_dirs = {}
for exp_key, exp_label in EXPERIMENT_LABELS.items():
    data[exp_key] = {}
    run_dirs[exp_key] = {}
    for basin in EXPERIMENT_BASINS[exp_key]:
        rd = find_latest_run(basin, exp_label)
        if rd is None:
            print(f'  MISSING: {basin} {exp_label}')
            continue
        run_dirs[exp_key][basin] = rd
        data[exp_key][basin] = {}
        for model in ['lstm', 'pilstm']:
            data[exp_key][basin][model] = {}
            for scale in ['1D', '1H']:
                data[exp_key][basin][model][scale] = load_ensemble(rd, model, scale)
        print(f'  {exp_key} {basin}: {rd.name}')

hms = {}
for basin in BASINS:
    hms[basin] = {}
    for scale in ['1D', '1H']:
        hms[basin][scale] = load_hms(basin, scale, extreme=False)
        hms[basin][f'{scale}_extreme'] = load_hms(basin, scale, extreme=True)

print('\nAll data loaded (predictions clipped to >= 0).')

## SQ1 - Model Performance Comparison (CV5)

In [ ]:
HATCHES = {'HMS': '', 'LSTM': '///', 'PILSTM': 'xxx'}

def compute_nse_kge(obs, pred):
    mask = ~(np.isnan(obs) | np.isnan(pred))
    o, p = obs[mask], pred[mask]
    nse = 1 - np.sum((o - p)**2) / np.sum((o - np.mean(o))**2)
    r = np.corrcoef(o, p)[0, 1]
    alpha = np.std(p) / np.std(o)
    beta = np.mean(p) / np.mean(o)
    kge = 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)
    return float(nse), float(kge)

rows = []
for basin in BASINS:
    for scale in ['1D', '1H']:
        freq_label = 'Daily' if scale == '1D' else 'Hourly'
        hms_df = hms[basin][scale]
        lstm_df = data['CV5'][basin]['lstm'][scale]
        pil_df = data['CV5'][basin]['pilstm'][scale]
        if lstm_df is None: continue
        merged = lstm_df.dropna(subset=['Observed', 'Predicted']).merge(hms_df, on='Date', how='inner')
        hms_nse, hms_kge = compute_nse_kge(merged['Observed'].values, merged['HMS'].values)
        lstm_nse, lstm_kge = compute_nse_kge(merged['Observed'].values, merged['Predicted'].values)
        pil_merged = pil_df.dropna(subset=['Observed', 'Predicted']).merge(hms_df, on='Date', how='inner')
        pil_nse, pil_kge = compute_nse_kge(pil_merged['Observed'].values, pil_merged['Predicted'].values)
        rows.append({'Basin': BASIN_LABELS[basin], 'Freq': freq_label, 'HMS_NSE': hms_nse, 'LSTM_NSE': lstm_nse, 'PILSTM_NSE': pil_nse, 'HMS_KGE': hms_kge, 'LSTM_KGE': lstm_kge, 'PILSTM_KGE': pil_kge})

metrics_df = pd.DataFrame(rows)

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharey=True)
bar_width = 0.25

for row_idx, freq in enumerate(['Daily', 'Hourly']):
    sub = metrics_df[metrics_df['Freq'] == freq]
    x = np.arange(len(sub))
    for col_idx, (metric, ylabel) in enumerate([('NSE', 'NSE'), ('KGE', 'KGE')]):
        ax = axes[row_idx, col_idx]
        for i, model in enumerate(MODELS):
            vals = sub[f'{model}_{metric}'].values
            xpos = x + i * bar_width
            ax.bar(xpos, vals, bar_width * 0.9, color=COLORS[model], edgecolor='black', linewidth=0.3, alpha=0.9, hatch=HATCHES[model], label=model if (row_idx == 0 and col_idx == 0) else None)
        ax.set_xticks(x + bar_width)
        ax.set_xticklabels(sub['Basin'].values)
        ax.set_ylabel(ylabel)
        ax.set_title(f'{freq} {metric}', fontweight='bold')
        ax.set_ylim(0.60, 1.02)
        ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.25, linewidth=0.8)
        ax.axhline(y=0.9, color='gray', linestyle='--', alpha=0.25, linewidth=0.8)
        ax.grid(axis='y', alpha=0.15)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.02), frameon=True, edgecolor='gray')
plt.tight_layout(rect=[0, 0, 1, 0.95])
save_path = SAVE_DIR / 'sq1_nse_kge_comparison.png'
fig.savefig(save_path)
plt.show()
print(f'Saved: {save_path}')
print(f'\nSQ1 Summary:')
print(metrics_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8))
bar_width = 0.11
gap = 0.06

for row_idx, freq in enumerate(['Daily', 'Hourly']):
    ax_nse = axes[row_idx]
    ax_kge = ax_nse.twinx()
    sub = metrics_df[metrics_df['Freq'] == freq]
    x = np.arange(len(sub))

    for i, model in enumerate(MODELS):
        nse_vals = sub[f'{model}_NSE'].values
        kge_vals = sub[f'{model}_KGE'].values

        xpos_nse = x + i * bar_width - (1.5 * bar_width + gap / 2)
        ax_nse.bar(xpos_nse, nse_vals, bar_width * 0.92, color=COLORS[model], alpha=0.9, edgecolor='black', linewidth=0.3, hatch=HATCHES[model])

        xpos_kge = x + i * bar_width + gap / 2
        ax_kge.bar(xpos_kge, kge_vals, bar_width * 0.92, color=COLORS[model], alpha=0.3, edgecolor=COLORS[model], linewidth=1.5, hatch='////')

    ax_nse.set_xticks(x)
    ax_nse.set_xticklabels(sub['Basin'].values, fontweight='bold')
    ax_nse.set_ylabel('NSE')
    ax_kge.set_ylabel('KGE')
    ax_nse.set_title(f'{freq}', fontweight='bold')
    ax_nse.set_ylim(0.60, 1.02)
    ax_kge.set_ylim(0.60, 1.02)
    ax_nse.axhline(y=0.8, color='gray', linestyle='--', alpha=0.2, linewidth=0.8)
    ax_nse.axhline(y=0.9, color='gray', linestyle='--', alpha=0.2, linewidth=0.8)
    ax_nse.grid(axis='y', alpha=0.1)
    ax_nse.set_zorder(ax_kge.get_zorder() + 1)
    ax_nse.patch.set_visible(False)

# 2-row legend
legend_handles = []
for model in MODELS:
    legend_handles.append(Patch(facecolor=COLORS[model], alpha=0.9, edgecolor='black', linewidth=0.3, hatch=HATCHES[model], label=f'{model} NSE'))
for model in MODELS:
    legend_handles.append(Patch(facecolor=COLORS[model], alpha=0.3, edgecolor=COLORS[model], linewidth=1.5, hatch='////', label=f'{model} KGE'))

fig.legend(handles=legend_handles, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.05), frameon=True, edgecolor='gray')
plt.tight_layout(rect=[0, 0.08, 1, 1])
save_path = SAVE_DIR / 'sq1_nse_kge_dual_axis.png'
fig.savefig(save_path)
plt.show()
print(f'Saved: {save_path}')

### SQ1 - Hydrographs (Full Test Period + Peak Zoom)

In [ ]:
from scipy.signal import find_peaks

def align_all_models(basin, scale, exp='CV5'):
    """Merge obs, HMS, LSTM, PILSTM into one DataFrame aligned by Date. All values clipped >= 0."""
    lstm_df = data[exp][basin]['lstm'][scale]
    pil_df = data[exp][basin]['pilstm'][scale]
    hms_key = f'{scale}_extreme' if exp == 'EXTREME' else scale
    hms_df = hms[basin][hms_key]
    if lstm_df is None: return None

    df = lstm_df[['Date', 'Observed', 'Predicted']].rename(columns={'Predicted': 'LSTM'}).copy()
    if pil_df is not None:
        df = df.merge(pil_df[['Date', 'Predicted']].rename(columns={'Predicted': 'PILSTM'}), on='Date', how='left')
    if hms_df is not None:
        df = df.merge(hms_df, on='Date', how='left')
    df = df.set_index('Date').sort_index()
    for col in ['Observed', 'LSTM', 'PILSTM', 'HMS']:
        if col in df.columns:
            df[col] = df[col].clip(lower=0)
    return df

def detect_peaks(obs_series, n_peaks=5, distance=30):
    """Find top N peaks in observed timeseries."""
    clean = obs_series.dropna()
    peaks_idx, props = find_peaks(clean.values, distance=distance, prominence=clean.std())
    peak_dates = clean.index[peaks_idx]
    peak_vals = clean.values[peaks_idx]
    top_idx = np.argsort(peak_vals)[::-1][:n_peaks]
    return peak_dates[top_idx], peak_vals[top_idx]

fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)
all_peaks = {}

for ax, basin in zip(axes, BASINS):
    df = align_all_models(basin, '1D', 'CV5')
    if df is None: continue

    ax.plot(df.index, df['Observed'], color='black', linewidth=1.3, alpha=0.9, label='Observed')
    for model, style in MODEL_STYLES.items():
        if model in df.columns:
            ax.plot(df.index, df[model], color=style['color'], linestyle=style['ls'], linewidth=style['lw'], alpha=0.7, label=model)

    peak_dates, peak_vals = detect_peaks(df['Observed'], n_peaks=5, distance=30)
    all_peaks[basin] = (peak_dates, peak_vals, df)
    ax.scatter(peak_dates, peak_vals, color='red', s=30, zorder=5, marker='v', alpha=0.7, label='Peak events')

    ax.set_ylim(bottom=0)
    ax.set_ylabel('Streamflow (CFS)')
    ax.set_title(BASIN_LABELS[basin], fontweight='bold', loc='left')
    ax.legend(ncol=5, loc='upper right')
    ax.grid(alpha=0.15)

axes[-1].set_xlabel('Date')
fig.suptitle('CV5 Test Period - Daily Hydrographs (SI)', fontweight='bold', y=1.01)
plt.tight_layout()
save_path = SAVE_DIR / 'si_hydrographs_daily_full.png'
fig.savefig(save_path)
plt.show()
print(f'Saved: {save_path}')

print('\nDetected peaks per basin:')
for basin, (dates, vals, _) in all_peaks.items():
    parts = [d.strftime('%Y-%m-%d') + ' (' + f'{v:.0f}' + ' CFS)' for d, v in zip(dates, vals)]
    print(f'  {BASIN_LABELS[basin]}: {", ".join(parts)}')

In [ ]:
# ---- SQ1: Peak zoom grid - DAILY + HOURLY (rows=basins, cols=top 2 events) ----

guern_dates, guern_vals = all_peaks['guerneville'][:2]
TOP_EVENTS = sorted(guern_dates[:2])
EVENT_LABELS = [d.strftime('%b %d, %Y') for d in TOP_EVENTS]
WINDOW = pd.Timedelta(days=12)

for scale, scale_label in [('1D', 'Daily'), ('1H', 'Hourly')]:
    fig, axes = plt.subplots(len(BASINS), len(TOP_EVENTS), figsize=(14, 13), sharey='row')

    for row_idx, basin in enumerate(BASINS):
        if scale == '1D':
            df = all_peaks[basin][2]
        else:
            df = align_all_models(basin, '1H', 'CV5')
        if df is None: continue

        for col_idx, (event_date, event_label) in enumerate(zip(TOP_EVENTS, EVENT_LABELS)):
            ax = axes[row_idx, col_idx]
            window_df = df.loc[event_date - WINDOW : event_date + WINDOW]

            if len(window_df) == 0 or window_df['Observed'].dropna().empty:
                ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
                continue

            ax.plot(window_df.index, window_df['Observed'], color='black', linewidth=2.2, alpha=0.9, label='Observed')
            for model, style in MODEL_STYLES.items():
                if model in window_df.columns:
                    ax.plot(window_df.index, window_df[model], color=style['color'], linestyle=style['ls'], linewidth=style['lw'], alpha=0.8, label=model)

            ax.set_ylim(bottom=0)
            ax.tick_params(axis='x', rotation=30)
            ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%m/%d'))
            ax.grid(alpha=0.15)

            if row_idx == 0:
                ax.set_title(event_label, fontweight='bold')
            if col_idx == 0:
                ax.set_ylabel('Streamflow (CFS)')
                ax.annotate(BASIN_LABELS[basin], xy=(-0.25, 0.5), xycoords='axes fraction', fontsize=13, fontweight='bold', rotation=90, va='center', ha='center')
            if row_idx == 0 and col_idx == 0:
                ax.legend(loc='upper right', framealpha=0.9)

    # Pad y-axis 10% above max per row
    for row_idx in range(len(BASINS)):
        row_max = 0
        for col_idx in range(len(TOP_EVENTS)):
            ax = axes[row_idx, col_idx]
            ymax = ax.get_ylim()[1]
            if ymax > row_max: row_max = ymax
        for col_idx in range(len(TOP_EVENTS)):
            axes[row_idx, col_idx].set_ylim(0, row_max * 1.10)

    fig.suptitle(f'CV5 Test Period - Peak Events ({scale_label})', fontweight='bold', y=1.01)
    plt.tight_layout()
    save_path = SAVE_DIR / f'sq1_peak_zoom_{scale_label.lower()}.png'
    fig.savefig(save_path)
    plt.show()
    print(f'Saved: {save_path}')

# Peak metrics table
print('\n--- Peak-by-Peak Metrics (Daily, Top 5 Events x 4 Basins) ---')
peak_rows = []
for basin in BASINS:
    df = all_peaks[basin][2]
    dates, vals = all_peaks[basin][:2]
    for peak_date, obs_peak in zip(dates, vals):
        row = {'Basin': BASIN_LABELS[basin], 'Date': peak_date.strftime('%Y-%m-%d'), 'Obs_Peak_CFS': obs_peak}
        window = df.loc[peak_date - pd.Timedelta(days=3) : peak_date + pd.Timedelta(days=3)]
        for model in MODELS:
            if model in window.columns and not window[model].isna().all():
                pred_peak = window[model].max()
                pred_date = window[model].idxmax()
                row[f'{model}_Peak'] = pred_peak
                row[f'{model}_Err%'] = (pred_peak - obs_peak) / obs_peak * 100
                row[f'{model}_Lag_d'] = (pred_date - peak_date).days
            else:
                row[f'{model}_Peak'] = np.nan
                row[f'{model}_Err%'] = np.nan
                row[f'{model}_Lag_d'] = np.nan
        peak_rows.append(row)

peak_df = pd.DataFrame(peak_rows)
print(peak_df.to_string(index=False, float_format='%.1f'))
peak_csv = SAVE_DIR / 'sq1_peak_metrics_daily.csv'
peak_df.to_csv(peak_csv, index=False, float_format='%.2f')
print(f'Saved: {peak_csv}')

print('\n--- Peak Error Summary ---')
for model in MODELS:
    errs = peak_df[f'{model}_Err%'].dropna()
    print(f'  {model}: mean err = {errs.mean():+.1f}%, median = {errs.median():+.1f}%, mean |err| = {errs.abs().mean():.1f}%')

### SQ1 - Scatter Plots (Observed vs Predicted)

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

SCATTER_AXIS_PERCENTILE = 99  # cap main axis at this percentile (99, 95, 90, or None for full range)
INSET_PERCENTILE = 95          # inset shows flows above this percentile

for scale, scale_label in [('1D', 'Daily'), ('1H', 'Hourly')]:
    fig, axes = plt.subplots(1, 4, figsize=(24, 6))

    for ax, basin in zip(axes, BASINS):
        df = align_all_models(basin, scale, 'CV5')
        if df is None: continue
        df_clean = df.dropna(subset=['Observed'])

        obs = df_clean['Observed'].values
        obs_valid = obs[~np.isnan(obs)]
        p_inset = np.percentile(obs_valid, INSET_PERCENTILE)

        # Axis limits
        if SCATTER_AXIS_PERCENTILE is not None:
            axis_cap = np.percentile(obs_valid, SCATTER_AXIS_PERCENTILE) * 1.1
            n_clipped = (obs_valid > axis_cap).sum()
        else:
            axis_cap = obs_valid.max() * 1.05
            n_clipped = 0
        lims = [0, axis_cap]

        # Main scatter
        for model in ['HMS', 'LSTM', 'PILSTM']:
            if model not in df_clean.columns: continue
            pred = df_clean[model].values
            mask = ~np.isnan(pred)
            ax.scatter(obs[mask], pred[mask], s=6, alpha=0.2, color=COLORS[model], label=model, rasterized=True)

        ax.plot(lims, lims, 'k-', linewidth=0.8, alpha=0.5, label='1:1')
        ax.set_xlim(lims)
        ax.set_ylim(lims)
        ax.set_xlabel('Observed (CFS)', fontsize=10)
        if basin == BASINS[0]:
            ax.set_ylabel('Predicted (CFS)', fontsize=10)
        ax.set_title(BASIN_LABELS[basin], fontsize=12, fontweight='bold')
        ax.set_aspect('equal')
        ax.grid(alpha=0.1)

        if n_clipped > 0:
            ax.text(0.97, 0.03, f'{n_clipped} pts beyond axis', transform=ax.transAxes, ha='right', va='bottom', fontsize=7, fontstyle='italic', color='gray')

        # Inset - top flows
        ax_ins = inset_axes(ax, width="45%", height="45%", loc='lower right', borderpad=2.0)
        high_mask = obs >= p_inset
        for model in ['HMS', 'LSTM', 'PILSTM']:
            if model not in df_clean.columns: continue
            pred = df_clean[model].values
            valid = high_mask & ~np.isnan(pred)
            ax_ins.scatter(obs[valid], pred[valid], s=14, alpha=0.5, color=COLORS[model], edgecolor='white', linewidth=0.3, rasterized=True)
        ins_max = obs[high_mask].max() * 1.1
        ins_min = p_inset * 0.8
        ax_ins.plot([ins_min, ins_max], [ins_min, ins_max], 'k-', linewidth=0.6, alpha=0.5)
        ax_ins.set_xlim(ins_min, ins_max)
        ax_ins.set_ylim(ins_min, ins_max)
        ax_ins.set_aspect('equal')
        ax_ins.tick_params(labelsize=7)
        ax_ins.set_title(f'Top {100 - INSET_PERCENTILE}%', fontsize=8, fontweight='bold')
        ax_ins.patch.set_facecolor('white')
        ax_ins.patch.set_alpha(0.95)
        for spine in ax_ins.spines.values():
            spine.set_edgecolor('gray')
            spine.set_linewidth(1.0)

    axes[0].legend(fontsize=9, loc='upper left', markerscale=3)
    fig.suptitle(f'CV5 Test Period - Observed vs Predicted ({scale_label})', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    save_path = SAVE_DIR / f'sq1_scatter_{scale_label.lower()}.png'
    fig.savefig(save_path)
    plt.show()
    print(f'Saved: {save_path}')

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

SCATTER_AXIS_PERCENTILE = 99
INSET_PERCENTILE = 95

for scale, scale_label in [('1D', 'Daily'), ('1H', 'Hourly')]:
    for basin in BASINS:
        df = align_all_models(basin, scale, 'CV5')
        if df is None: continue
        df_clean = df.dropna(subset=['Observed'])

        obs = df_clean['Observed'].values
        obs_valid = obs[~np.isnan(obs)]
        p_inset = np.percentile(obs_valid, INSET_PERCENTILE)

        if SCATTER_AXIS_PERCENTILE is not None:
            axis_cap = np.percentile(obs_valid, SCATTER_AXIS_PERCENTILE) * 1.1
            n_clipped = (obs_valid > axis_cap).sum()
        else:
            axis_cap = obs_valid.max() * 1.05
            n_clipped = 0
        lims = [0, axis_cap]

        fig, ax = plt.subplots(1, 1, figsize=(7, 7))
        marker_s = 6 if scale == '1D' else 3
        marker_alpha = 0.15 if scale == '1D' else 0.08

        for model in MODELS:
            if model not in df_clean.columns: continue
            pred = df_clean[model].values
            mask = ~np.isnan(pred)
            ax.scatter(obs[mask], pred[mask], s=marker_s, alpha=marker_alpha, color=COLORS[model], label=model, rasterized=True)

        ax.plot(lims, lims, 'k-', linewidth=0.8, alpha=0.5, label='1:1')
        ax.set_xlim(lims)
        ax.set_ylim(lims)
        ax.set_xlabel('Observed Streamflow (CFS)')
        ax.set_ylabel('Predicted Streamflow (CFS)')
        ax.set_title(f'{BASIN_LABELS[basin]} - {scale_label}', fontweight='bold')
        ax.set_aspect('equal')
        ax.grid(alpha=0.1)
        ax.legend(markerscale=3, loc='upper left')

        if n_clipped > 0:
            ax.text(0.97, 0.97, f'{n_clipped} pts beyond axis', transform=ax.transAxes, ha='right', va='top', fontsize=9, fontstyle='italic', color='#555555')

        # Inset
        ax_ins = inset_axes(ax, width="40%", height="40%", loc='lower right', borderpad=2.5)
        high_mask = obs >= p_inset
        for model in MODELS:
            if model not in df_clean.columns: continue
            pred = df_clean[model].values
            valid = high_mask & ~np.isnan(pred)
            ax_ins.scatter(obs[valid], pred[valid], s=16, alpha=0.5, color=COLORS[model], edgecolor='white', linewidth=0.3, rasterized=True)
        ins_max = obs[high_mask].max() * 1.1
        ins_min = p_inset * 0.8
        ax_ins.plot([ins_min, ins_max], [ins_min, ins_max], 'k-', linewidth=0.6, alpha=0.5)
        ax_ins.set_xlim(ins_min, ins_max)
        ax_ins.set_ylim(ins_min, ins_max)
        ax_ins.set_aspect('equal')
        ax_ins.tick_params(labelsize=9)
        ax_ins.locator_params(nbins=4)
        ax_ins.set_title(f'Top {100 - INSET_PERCENTILE}%', fontsize=10, fontweight='bold')
        ax_ins.patch.set_facecolor('white')
        ax_ins.patch.set_alpha(0.95)
        for spine in ax_ins.spines.values():
            spine.set_edgecolor('gray')
            spine.set_linewidth(1.0)

        plt.tight_layout()
        basin_tag = basin.replace('_', '')
        save_path = SAVE_DIR / f'sq1_scatter_{basin_tag}_{scale_label.lower()}.png'
        fig.savefig(save_path)
        plt.show()
        print(f'Saved: {save_path}')

In [ ]:
# ---- SQ1: Individual FDC per basin (daily + hourly) ----

def compute_fdc(series):
    """Return exceedance probability and sorted flows."""
    clean = series.dropna().values
    sorted_flows = np.sort(clean)[::-1]
    n = len(sorted_flows)
    exceedance = np.arange(1, n + 1) / (n + 1) * 100
    return exceedance, sorted_flows

for scale, scale_label in [('1D', 'Daily'), ('1H', 'Hourly')]:
    for basin in BASINS:
        df = align_all_models(basin, scale, 'CV5')
        if df is None: continue

        fig, ax = plt.subplots(1, 1, figsize=(8, 6))

        exc_obs, flows_obs = compute_fdc(df['Observed'])
        ax.plot(exc_obs, flows_obs, color='black', linewidth=2.2, alpha=0.9, label='Observed')

        for model, style in MODEL_STYLES.items():
            if model not in df.columns: continue
            exc, flows = compute_fdc(df[model])
            ax.plot(exc, flows, color=style['color'], linestyle=style['ls'], linewidth=style['lw'], alpha=0.8, label=model)

        ax.set_yscale('log')
        ax.set_xlabel('Exceedance Probability (%)')
        ax.set_ylabel('Streamflow (CFS)')
        ax.set_title(f'{BASIN_LABELS[basin]} - {scale_label}', fontweight='bold')
        ax.set_xlim(0, 100)
        ax.grid(alpha=0.15, which='both')

        # FHV: vertical line at 2% (narrow shading was invisible)
        ax.axvline(x=2, color='red', linestyle=':', linewidth=1.0, alpha=0.5)
        # FMS: shaded region with bumped alpha
        ax.axvspan(20, 70, alpha=0.08, color='gray')
        # FLV: shaded region
        ax.axvspan(70, 100, alpha=0.08, color='#90CAF9')

        handles, labels = ax.get_legend_handles_labels()
        handles += [plt.Line2D([0], [0], color='red', linestyle=':', linewidth=1.0, alpha=0.5, label='FHV (2%)'),
                    Patch(facecolor='gray', alpha=0.15, label='FMS (20-70%)'),
                    Patch(facecolor='#90CAF9', alpha=0.15, label='FLV (70-100%)')]
        ax.legend(handles=handles, loc='upper right')

        plt.tight_layout()
        basin_tag = basin.replace('_', '')
        save_path = SAVE_DIR / f'sq1_fdc_{basin_tag}_{scale_label.lower()}.png'
        fig.savefig(save_path)
        plt.show()
        print(f'Saved: {save_path}')

### SQ1 - Integrated Gradients Attribution (CV5, all members averaged)

In [ ]:
# ---- SQ1: IG Attribution - SKIP (already computed, CSVs saved) ----
# Uncomment and run to recompute IG attributions (takes ~1hr)
# Results at: SAVE_DIR / 'sq1_ig_{basin}_{model}_{freq}.csv'

"""
from UCB_training.ucb_captum import run_ig_analysis_mts
... (full IG computation code - already ran, 16 CSVs saved)
"""

# Load saved IG results instead
ig_results = {}
for basin in BASINS:
    for model_label in ['lstm', 'pilstm']:
        for freq in ['1d', '1h']:
            csv_path = SAVE_DIR / f'sq1_ig_{basin.replace("_", "")}_{model_label}_{freq}.csv'
            if csv_path.exists():
                ig_results[(basin, model_label.upper(), freq.upper())] = pd.read_csv(csv_path)

print(f'Loaded {len(ig_results)} IG result CSVs')
for key in sorted(ig_results.keys()):
    df = ig_results[key]
    print(f'  {key}: top feature = {df.iloc[0]["feature"]} ({df.iloc[0]["mean_abs_attr"]:.6f})')

In [ ]:
TOP_N = 10

def abbrev_feature(name):
    replacements = [
        ('ET-POTENTIAL RUN:BASIN AVERAGE 60 YR', 'ET-Pot'),
        ('PRECIP-INC SCREENED', 'Precip'),
        ('SATURATION FRACTION', 'Sat-Frac'),
        ('INFILTRATION', 'Infilt'),
        ('PERC-SOIL', 'Perc'),
        ('FLOW-BASE', 'Baseflow'),
        ('FLOW', 'Flow'),
        ('ET-POTENTIAL', 'ET-Pot'),
        ('USGS-MERGED', 'USGS'),
        ('USGS_ADJUSTED', 'USGS'),
        ('USAF-NOAA', ''),
        ('HUMIDITY', 'Humid'),
        ('TEMPERATURE', 'Temp'),
        ('SOLAR RADIATION', 'Solar'),
        ('WINDSPEED', 'Wind'),
        ('SANTA ROSA', 'S.Rosa'),
        ('RUSSIAN', 'Rus'),
        ('DRY CREEK', 'DryCreek'),
        ('BIG SULPHUR CR', 'BigSulph'),
        ('GREEN VALLEY', 'GreenVal'),
        ('EF RUSSIAN', 'EF Rus'),
        ('WF RUSSIAN', 'WF Rus'),
        ('POTTER VALLEY CA', 'PotterVal'),
        ('UKIAH CA', 'Ukiah'),
        ('  ', ' '),
    ]
    s = name
    for old, new in replacements:
        s = s.replace(old, new)
    return s.strip()

for freq, freq_label in [('1D', 'Daily'), ('1H', 'Hourly')]:
    for model_label in ['LSTM', 'PILSTM']:
        all_features = set()
        basin_dfs = {}
        for basin in BASINS:
            key = (basin, model_label, freq)
            if key not in ig_results: continue
            df = ig_results[key].head(TOP_N)
            all_features.update(df['feature'].tolist())
            basin_dfs[basin] = ig_results[key]

        if not basin_dfs: continue

        all_features = sorted(all_features)
        basin_cols = [BASIN_LABELS[b] for b in BASINS if b in basin_dfs]
        heat_data = pd.DataFrame(index=all_features, columns=basin_cols)
        for basin, df_ig in basin_dfs.items():
            feat_map = dict(zip(df_ig['feature'], df_ig['mean_abs_attr']))
            for feat in all_features:
                heat_data.loc[feat, BASIN_LABELS[basin]] = feat_map.get(feat, 0.0)

        heat_data = heat_data.astype(float)
        heat_data['mean'] = heat_data.mean(axis=1)
        heat_data = heat_data.sort_values('mean', ascending=True).drop(columns='mean')

        abbrev_idx = [abbrev_feature(f) for f in heat_data.index]

        fig, ax = plt.subplots(1, 1, figsize=(8, max(6, len(heat_data) * 0.35)))
        im = ax.imshow(heat_data.values, aspect='auto', cmap='YlOrRd', interpolation='nearest')

        ax.set_xticks(range(len(heat_data.columns)))
        ax.set_xticklabels(heat_data.columns, fontweight='bold')
        ax.set_yticks(range(len(abbrev_idx)))
        ax.set_yticklabels(abbrev_idx, fontsize=9)
        ax.set_title(f'{model_label} - {freq_label} IG Attribution (Top {TOP_N} per basin)', fontweight='bold')
        plt.colorbar(im, ax=ax, label='Mean |Attribution|', shrink=0.8)

        # White gridlines between cells
        for edge in range(len(heat_data.columns) + 1):
            ax.axvline(edge - 0.5, color='white', linewidth=0.5)
        for edge in range(len(heat_data.index) + 1):
            ax.axhline(edge - 0.5, color='white', linewidth=0.5)

        plt.tight_layout()
        save_path = SAVE_DIR / f'sq1_ig_heatmap_{model_label.lower()}_{freq_label.lower()}.png'
        fig.savefig(save_path)
        plt.show()
        print(f'Saved: {save_path}')

print('\n--- FLOW feature importance (attribution != necessity) ---')
for basin in BASINS:
    for model in ['LSTM', 'PILSTM']:
        for freq in ['1D', '1H']:
            key = (basin, model, freq)
            if key not in ig_results: continue
            df = ig_results[key]
            flow_feats = df[df['feature'].str.contains('FLOW', case=False)]
            if not flow_feats.empty:
                top_flow = flow_feats.iloc[0]
                total_rank = len(df)
                print(f'  {BASIN_LABELS[basin]} {model} {freq}: top FLOW = {abbrev_feature(top_flow["feature"])} (rank {int(top_flow["rank"])}/{total_rank})')

### SQ1 - Flow Duration Curves

In [ ]:
for scale, scale_label in [('1D', 'Daily'), ('1H', 'Hourly')]:
    fig, axes = plt.subplots(1, 4, figsize=(24, 6))

    for ax, basin in zip(axes, BASINS):
        df = align_all_models(basin, scale, 'CV5')
        if df is None: continue

        exc_obs, flows_obs = compute_fdc(df['Observed'])
        ax.plot(exc_obs, flows_obs, color='black', linewidth=2.2, alpha=0.9, label='Observed')

        for model, style in MODEL_STYLES.items():
            if model not in df.columns: continue
            exc, flows = compute_fdc(df[model])
            ax.plot(exc, flows, color=style['color'], linestyle=style['ls'], linewidth=style['lw'], alpha=0.8, label=model)

        ax.set_yscale('log')
        ax.set_xlabel('Exceedance Probability (%)')
        if basin == BASINS[0]:
            ax.set_ylabel('Streamflow (CFS)')
        ax.set_title(BASIN_LABELS[basin], fontweight='bold')
        ax.set_xlim(0, 100)
        ax.grid(alpha=0.15, which='both')

        ax.axvline(x=2, color='red', linestyle=':', linewidth=1.0, alpha=0.5)
        ax.axvspan(20, 70, alpha=0.08, color='gray')
        ax.axvspan(70, 100, alpha=0.08, color='#90CAF9')

        handles, labels = ax.get_legend_handles_labels()
        handles += [plt.Line2D([0], [0], color='red', linestyle=':', linewidth=1.0, alpha=0.5, label='FHV'),
                    Patch(facecolor='gray', alpha=0.15, label='FMS'),
                    Patch(facecolor='#90CAF9', alpha=0.15, label='FLV')]
        ax.legend(handles=handles, fontsize=8, loc='upper right')

    fig.suptitle(f'CV5 Test Period - Flow Duration Curves ({scale_label})', fontweight='bold', y=1.02)
    plt.tight_layout()
    save_path = SAVE_DIR / f'sq1_fdc_{scale_label.lower()}.png'
    fig.savefig(save_path)
    plt.show()
    print(f'Saved: {save_path}')

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)

for ax, basin in zip(axes, BASINS):
    df = align_all_models(basin, '1H', 'CV5')
    if df is None: continue

    ax.plot(df.index, df['Observed'], color='black', linewidth=0.6, alpha=0.7, label='Observed')
    for model, style in MODEL_STYLES.items():
        if model in df.columns:
            ax.plot(df.index, df[model], color=style['color'], linestyle=style['ls'], linewidth=0.5, alpha=0.6, label=model)

    ax.set_ylim(bottom=0)
    ax.set_ylabel('Streamflow (CFS)')
    ax.set_title(BASIN_LABELS[basin], fontweight='bold', loc='left')
    ax.legend(ncol=4, loc='upper right')
    ax.grid(alpha=0.15)

axes[-1].set_xlabel('Date')
fig.suptitle('CV5 Test Period - Hourly Hydrographs (SI)', fontweight='bold', y=1.01)
plt.tight_layout()
save_path = SAVE_DIR / 'si_hydrographs_hourly_full.png'
fig.savefig(save_path)
plt.show()
print(f'Saved: {save_path}')

## SQ2 - Extreme Holdout Results

In [ ]:
sq2_rows = []
for exp_key, exp_label in [('CV5', 'Standard'), ('EXTREME', 'Extreme')]:
    for basin in BASINS:
        if basin not in data[exp_key]: continue
        for scale in ['1D', '1H']:
            freq_label = 'Daily' if scale == '1D' else 'Hourly'
            hms_key = f'{scale}_extreme' if exp_key == 'EXTREME' else scale
            hms_df = hms[basin][hms_key]
            lstm_df = data[exp_key][basin]['lstm'][scale]
            pil_df = data[exp_key][basin]['pilstm'][scale]
            if lstm_df is None: continue
            merged = lstm_df.dropna(subset=['Observed', 'Predicted']).merge(hms_df, on='Date', how='inner')
            hms_nse, hms_kge = compute_nse_kge(merged['Observed'].values, merged['HMS'].values)
            lstm_nse, lstm_kge = compute_nse_kge(merged['Observed'].values, merged['Predicted'].values)
            pil_merged = pil_df.dropna(subset=['Observed', 'Predicted']).merge(hms_df, on='Date', how='inner')
            pil_nse, pil_kge = compute_nse_kge(pil_merged['Observed'].values, pil_merged['Predicted'].values)
            blend_nse, blend_kge = np.nan, np.nan
            if exp_key == 'EXTREME':
                blend_pred = (merged['Predicted'].values + merged['HMS'].values) / 2.0
                blend_nse, blend_kge = compute_nse_kge(merged['Observed'].values, blend_pred)
            sq2_rows.append({'Experiment': exp_label, 'Basin': BASIN_LABELS[basin], 'Freq': freq_label,
                             'HMS_NSE': hms_nse, 'LSTM_NSE': lstm_nse, 'PILSTM_NSE': pil_nse, 'Blend_NSE': blend_nse,
                             'HMS_KGE': hms_kge, 'LSTM_KGE': lstm_kge, 'PILSTM_KGE': pil_kge, 'Blend_KGE': blend_kge})

sq2_df = pd.DataFrame(sq2_rows)

COLORS_SQ2 = {**COLORS, 'HMS+LSTM': '#E53935'}
MODELS_SQ2 = ['HMS', 'LSTM', 'PILSTM', 'HMS+LSTM']
HATCHES_SQ2 = {**HATCHES, 'HMS+LSTM': '...'}

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
bar_width = 0.09
gap = 0.05

for row_idx, freq in enumerate(['Daily', 'Hourly']):
    ax_nse = axes[row_idx]
    ax_kge = ax_nse.twinx()
    sub = sq2_df[(sq2_df['Freq'] == freq) & (sq2_df['Experiment'] == 'Extreme')]
    x = np.arange(len(sub))

    for i, (model, col_suffix) in enumerate(zip(MODELS_SQ2, ['HMS', 'LSTM', 'PILSTM', 'Blend'])):
        nse_vals = sub[f'{col_suffix}_NSE'].values
        kge_vals = sub[f'{col_suffix}_KGE'].values
        color = COLORS_SQ2[model]

        xpos_nse = x + i * bar_width - (2 * bar_width + gap / 2)
        ax_nse.bar(xpos_nse, nse_vals, bar_width * 0.9, color=color, alpha=0.9, edgecolor='black', linewidth=0.3, hatch=HATCHES_SQ2[model])

        xpos_kge = x + i * bar_width + gap / 2
        ax_kge.bar(xpos_kge, kge_vals, bar_width * 0.9, color=color, alpha=0.3, edgecolor=color, linewidth=1.5, hatch='////')

    ax_nse.set_xticks(x)
    ax_nse.set_xticklabels(sub['Basin'].values, fontweight='bold')
    ax_nse.set_ylabel('NSE')
    ax_kge.set_ylabel('KGE')
    ax_nse.set_title(f'{freq}', fontweight='bold')
    ax_nse.set_ylim(0.60, 1.02)
    ax_kge.set_ylim(0.60, 1.02)
    ax_nse.axhline(y=0.8, color='gray', linestyle='--', alpha=0.2, linewidth=0.8)
    ax_nse.axhline(y=0.9, color='gray', linestyle='--', alpha=0.2, linewidth=0.8)
    ax_nse.grid(axis='y', alpha=0.1)
    ax_nse.set_zorder(ax_kge.get_zorder() + 1)
    ax_nse.patch.set_visible(False)

legend_handles = []
for model in MODELS_SQ2:
    legend_handles.append(Patch(facecolor=COLORS_SQ2[model], alpha=0.9, edgecolor='black', linewidth=0.3, hatch=HATCHES_SQ2[model], label=f'{model} NSE'))
for model in MODELS_SQ2:
    legend_handles.append(Patch(facecolor=COLORS_SQ2[model], alpha=0.3, edgecolor=COLORS_SQ2[model], linewidth=1.5, hatch='////', label=f'{model} KGE'))

fig.legend(handles=legend_handles, loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.06), frameon=True, edgecolor='gray')
plt.tight_layout(rect=[0, 0.08, 1, 1])
save_path = SAVE_DIR / 'sq2_extreme_nse_kge.png'
fig.savefig(save_path)
plt.show()
print(f'Saved: {save_path}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for col_idx, freq in enumerate(['Daily', 'Hourly']):
    ax = axes[col_idx]
    std = sq2_df[(sq2_df['Freq'] == freq) & (sq2_df['Experiment'] == 'Standard')].set_index('Basin')
    ext = sq2_df[(sq2_df['Freq'] == freq) & (sq2_df['Experiment'] == 'Extreme')].set_index('Basin')
    basins_common = std.index.intersection(ext.index)

    x = np.arange(len(basins_common))
    bar_w = 0.25

    for i, model in enumerate(MODELS):
        delta = ext.loc[basins_common, f'{model}_NSE'].values - std.loc[basins_common, f'{model}_NSE'].values
        ax.bar(x + i * bar_w, delta, bar_w * 0.9, color=COLORS[model], edgecolor='black', linewidth=0.3, alpha=0.9, hatch=HATCHES[model], label=model if col_idx == 0 else None)
        for xi, d in zip(x + i * bar_w, delta):
            offset = 0.008 if d >= 0 else -0.008
            ax.text(xi, d + offset, f'{d:+.3f}', ha='center', va='bottom' if d >= 0 else 'top', fontsize=8, fontweight='bold')

    ax.axhline(y=0, color='black', linewidth=0.8)
    ax.set_xticks(x + bar_w)
    ax.set_xticklabels(basins_common, fontweight='bold')
    ax.set_ylabel('Delta NSE (Extreme - Standard)')
    ax.set_title(f'{freq}', fontweight='bold')
    ax.grid(axis='y', alpha=0.1)
    ax.set_ylim(ax.get_ylim()[0] - 0.02, ax.get_ylim()[1] + 0.02)  # padding for labels

axes[0].legend(loc='lower left')
fig.suptitle('NSE Change: Standard to Extreme Holdout', fontweight='bold', y=1.01)
plt.tight_layout()
save_path = SAVE_DIR / 'sq2_delta_nse.png'
fig.savefig(save_path)
plt.show()
print(f'Saved: {save_path}')

print('\n--- SQ2 Summary ---')
print(sq2_df.to_string(index=False, float_format='%.4f'))

In [ ]:
all_peaks_ext = {}
for basin in BASINS:
    df = align_all_models(basin, '1D', 'EXTREME')
    if df is None: continue
    peak_dates, peak_vals = detect_peaks(df['Observed'], n_peaks=5, distance=30)
    all_peaks_ext[basin] = (peak_dates, peak_vals, df)
    parts = [d.strftime('%Y-%m-%d') + ' (' + f'{v:.0f}' + ' CFS)' for d, v in zip(peak_dates, peak_vals)]
    print(f'  {BASIN_LABELS[basin]}: {", ".join(parts)}')

guern_ext_dates, guern_ext_vals = all_peaks_ext['guerneville'][:2]
TOP_EXT_EVENTS = sorted(guern_ext_dates[:2])
EXT_EVENT_LABELS = [d.strftime('%b %d, %Y') for d in TOP_EXT_EVENTS]
WINDOW = pd.Timedelta(days=12)
BLEND_COLOR = '#E53935'

for scale, scale_label in [('1D', 'Daily'), ('1H', 'Hourly')]:
    fig, axes = plt.subplots(len(BASINS), len(TOP_EXT_EVENTS), figsize=(14, 13), sharey='row')

    for row_idx, basin in enumerate(BASINS):
        if scale == '1D':
            df = all_peaks_ext[basin][2]
        else:
            df = align_all_models(basin, '1H', 'EXTREME')
        if df is None: continue

        for col_idx, (event_date, event_label) in enumerate(zip(TOP_EXT_EVENTS, EXT_EVENT_LABELS)):
            ax = axes[row_idx, col_idx]
            window_df = df.loc[event_date - WINDOW : event_date + WINDOW]

            if len(window_df) == 0 or window_df['Observed'].dropna().empty:
                ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
                continue

            ax.plot(window_df.index, window_df['Observed'], color='black', linewidth=2.2, alpha=0.9, label='Observed')
            for model, style in MODEL_STYLES.items():
                if model in window_df.columns:
                    ax.plot(window_df.index, window_df[model], color=style['color'], linestyle=style['ls'], linewidth=style['lw'], alpha=0.8, label=model)

            # HMS+LSTM blend
            if 'HMS' in window_df.columns and 'LSTM' in window_df.columns:
                blend = (window_df['HMS'] + window_df['LSTM']) / 2.0
                ax.plot(window_df.index, blend, color=BLEND_COLOR, linestyle='-', linewidth=1.8, alpha=0.85, label='HMS+LSTM')

            ax.set_ylim(bottom=0)
            ax.tick_params(axis='x', rotation=30)
            ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%m/%d'))
            ax.grid(alpha=0.15)

            if row_idx == 0:
                ax.set_title(event_label, fontweight='bold')
            if col_idx == 0:
                ax.set_ylabel('Streamflow (CFS)')
                ax.annotate(BASIN_LABELS[basin], xy=(-0.25, 0.5), xycoords='axes fraction', fontsize=13, fontweight='bold', rotation=90, va='center', ha='center')
            if row_idx == 0 and col_idx == 0:
                ax.legend(loc='upper right', framealpha=0.9, fontsize=8)

    # Pad y-axis 10% above max per row to prevent clipping
    for row_idx in range(len(BASINS)):
        row_max = 0
        for col_idx in range(len(TOP_EXT_EVENTS)):
            ax = axes[row_idx, col_idx]
            ymax = ax.get_ylim()[1]
            if ymax > row_max: row_max = ymax
        for col_idx in range(len(TOP_EXT_EVENTS)):
            axes[row_idx, col_idx].set_ylim(0, row_max * 1.10)

    fig.suptitle(f'Extreme Holdout - Peak Events ({scale_label})', fontweight='bold', y=1.01)
    plt.tight_layout()
    save_path = SAVE_DIR / f'sq2_peak_zoom_{scale_label.lower()}.png'
    fig.savefig(save_path)
    plt.show()
    print(f'Saved: {save_path}')

print('\n--- Peak-by-Peak Metrics (Extreme Daily) ---')
ext_peak_rows = []
for basin in BASINS:
    df = all_peaks_ext[basin][2]
    dates, vals = all_peaks_ext[basin][:2]
    for peak_date, obs_peak in zip(dates, vals):
        row = {'Basin': BASIN_LABELS[basin], 'Date': peak_date.strftime('%Y-%m-%d'), 'Obs_Peak_CFS': obs_peak}
        window = df.loc[peak_date - pd.Timedelta(days=3) : peak_date + pd.Timedelta(days=3)]
        for model in MODELS:
            if model in window.columns and not window[model].isna().all():
                pred_peak = window[model].max()
                pred_date = window[model].idxmax()
                row[f'{model}_Peak'] = pred_peak
                row[f'{model}_Err%'] = (pred_peak - obs_peak) / obs_peak * 100
                row[f'{model}_Lag_d'] = (pred_date - peak_date).days
            else:
                row[f'{model}_Peak'] = np.nan
                row[f'{model}_Err%'] = np.nan
                row[f'{model}_Lag_d'] = np.nan
        # Blend peak
        if 'HMS' in window.columns and 'LSTM' in window.columns:
            blend_ts = (window['HMS'] + window['LSTM']) / 2.0
            blend_peak = blend_ts.max()
            blend_date = blend_ts.idxmax()
            row['Blend_Peak'] = blend_peak
            row['Blend_Err%'] = (blend_peak - obs_peak) / obs_peak * 100
            row['Blend_Lag_d'] = (blend_date - peak_date).days
        ext_peak_rows.append(row)

ext_peak_df = pd.DataFrame(ext_peak_rows)
print(ext_peak_df.to_string(index=False, float_format='%.1f'))
ext_peak_csv = SAVE_DIR / 'sq2_peak_metrics_extreme_daily.csv'
ext_peak_df.to_csv(ext_peak_csv, index=False, float_format='%.2f')
print(f'Saved: {ext_peak_csv}')

print('\n--- Extreme Peak Error Summary ---')
for model in MODELS + ['Blend']:
    col = f'{model}_Err%'
    if col in ext_peak_df.columns:
        errs = ext_peak_df[col].dropna()
        print(f'  {model}: mean err = {errs.mean():+.1f}%, median = {errs.median():+.1f}%, mean |err| = {errs.abs().mean():.1f}%')

### SQ2 - Scatter Plots + FDC + Blend Hydrograph (Extreme)

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

SCATTER_AXIS_PERCENTILE = 99
INSET_PERCENTILE = 95

for scale, scale_label in [('1D', 'Daily'), ('1H', 'Hourly')]:
    for basin in BASINS:
        df = align_all_models(basin, scale, 'EXTREME')
        if df is None: continue
        df_clean = df.dropna(subset=['Observed'])

        obs = df_clean['Observed'].values
        obs_valid = obs[~np.isnan(obs)]
        p_inset = np.percentile(obs_valid, INSET_PERCENTILE)

        if SCATTER_AXIS_PERCENTILE is not None:
            axis_cap = np.percentile(obs_valid, SCATTER_AXIS_PERCENTILE) * 1.1
            n_clipped = (obs_valid > axis_cap).sum()
        else:
            axis_cap = obs_valid.max() * 1.05
            n_clipped = 0
        lims = [0, axis_cap]

        fig, ax = plt.subplots(1, 1, figsize=(7, 7))
        marker_s = 6 if scale == '1D' else 3
        marker_alpha = 0.15 if scale == '1D' else 0.08

        for model in MODELS:
            if model not in df_clean.columns: continue
            pred = df_clean[model].values
            mask = ~np.isnan(pred)
            ax.scatter(obs[mask], pred[mask], s=marker_s, alpha=marker_alpha, color=COLORS[model], label=model, rasterized=True)

        ax.plot(lims, lims, 'k-', linewidth=0.8, alpha=0.5, label='1:1')
        ax.set_xlim(lims)
        ax.set_ylim(lims)
        ax.set_xlabel('Observed Streamflow (CFS)')
        ax.set_ylabel('Predicted Streamflow (CFS)')
        ax.set_title(f'{BASIN_LABELS[basin]} - Extreme {scale_label}', fontweight='bold')
        ax.set_aspect('equal')
        ax.grid(alpha=0.1)
        ax.legend(markerscale=3, loc='upper left')

        if n_clipped > 0:
            ax.text(0.97, 0.97, f'{n_clipped} pts beyond axis', transform=ax.transAxes, ha='right', va='top', fontsize=9, fontstyle='italic', color='#555555')

        ax_ins = inset_axes(ax, width="40%", height="40%", loc='lower right', borderpad=2.5)
        high_mask = obs >= p_inset
        for model in MODELS:
            if model not in df_clean.columns: continue
            pred = df_clean[model].values
            valid = high_mask & ~np.isnan(pred)
            ax_ins.scatter(obs[valid], pred[valid], s=16, alpha=0.5, color=COLORS[model], edgecolor='white', linewidth=0.3, rasterized=True)
        ins_max = obs[high_mask].max() * 1.1
        ins_min = p_inset * 0.8
        ax_ins.plot([ins_min, ins_max], [ins_min, ins_max], 'k-', linewidth=0.6, alpha=0.5)
        ax_ins.set_xlim(ins_min, ins_max)
        ax_ins.set_ylim(ins_min, ins_max)
        ax_ins.set_aspect('equal')
        ax_ins.tick_params(labelsize=9)
        ax_ins.locator_params(nbins=4)
        ax_ins.set_title(f'Top {100 - INSET_PERCENTILE}%', fontsize=10, fontweight='bold')
        ax_ins.patch.set_facecolor('white')
        ax_ins.patch.set_alpha(0.95)
        for spine in ax_ins.spines.values():
            spine.set_edgecolor('gray')
            spine.set_linewidth(1.0)

        plt.tight_layout()
        basin_tag = basin.replace('_', '')
        save_path = SAVE_DIR / f'sq2_scatter_{basin_tag}_{scale_label.lower()}.png'
        fig.savefig(save_path)
        plt.show()
        print(f'Saved: {save_path}')

In [ ]:
for scale, scale_label in [('1D', 'Daily'), ('1H', 'Hourly')]:
    for basin in BASINS:
        df = align_all_models(basin, scale, 'EXTREME')
        if df is None: continue

        fig, ax = plt.subplots(1, 1, figsize=(8, 6))

        exc_obs, flows_obs = compute_fdc(df['Observed'])
        ax.plot(exc_obs, flows_obs, color='black', linewidth=2.2, alpha=0.9, label='Observed')

        for model, style in MODEL_STYLES.items():
            if model not in df.columns: continue
            exc, flows = compute_fdc(df[model])
            ax.plot(exc, flows, color=style['color'], linestyle=style['ls'], linewidth=style['lw'], alpha=0.8, label=model)

        ax.set_yscale('log')
        ax.set_xlabel('Exceedance Probability (%)')
        ax.set_ylabel('Streamflow (CFS)')
        ax.set_title(f'{BASIN_LABELS[basin]} - Extreme {scale_label}', fontweight='bold')
        ax.set_xlim(0, 100)
        ax.grid(alpha=0.15, which='both')

        ax.axvline(x=2, color='red', linestyle=':', linewidth=1.0, alpha=0.5)
        ax.axvspan(20, 70, alpha=0.08, color='gray')
        ax.axvspan(70, 100, alpha=0.08, color='#90CAF9')

        handles, labels = ax.get_legend_handles_labels()
        handles += [plt.Line2D([0], [0], color='red', linestyle=':', linewidth=1.0, alpha=0.5, label='FHV (2%)'),
                    Patch(facecolor='gray', alpha=0.15, label='FMS (20-70%)'),
                    Patch(facecolor='#90CAF9', alpha=0.15, label='FLV (70-100%)')]
        ax.legend(handles=handles, loc='upper right')

        plt.tight_layout()
        basin_tag = basin.replace('_', '')
        save_path = SAVE_DIR / f'sq2_fdc_{basin_tag}_{scale_label.lower()}.png'
        fig.savefig(save_path)
        plt.show()
        print(f'Saved: {save_path}')

In [ ]:
BLEND_COLOR = '#E53935'  # red

for scale, scale_label in [('1D', 'Daily'), ('1H', 'Hourly')]:
    fig, axes = plt.subplots(len(BASINS), 1, figsize=(14, 13))

    for ax, basin in zip(axes, BASINS):
        df = align_all_models(basin, scale, 'EXTREME')
        if df is None: continue

        # Find biggest peak for this basin
        peak_dates_ext, peak_vals_ext = detect_peaks(df['Observed'], n_peaks=1, distance=30)
        if len(peak_dates_ext) == 0: continue
        event = peak_dates_ext[0]
        window = pd.Timedelta(days=12)
        wdf = df.loc[event - window : event + window]

        if wdf['Observed'].dropna().empty: continue

        # Plot observed + 3 models + blend
        ax.plot(wdf.index, wdf['Observed'], color='black', linewidth=2.5, alpha=0.9, label='Observed')
        for model, style in MODEL_STYLES.items():
            if model in wdf.columns:
                ax.plot(wdf.index, wdf[model], color=style['color'], linestyle=style['ls'], linewidth=style['lw'], alpha=0.7, label=model)

        # Blend
        if 'HMS' in wdf.columns and 'LSTM' in wdf.columns:
            blend = (wdf['HMS'] + wdf['LSTM']) / 2.0
            ax.plot(wdf.index, blend, color=BLEND_COLOR, linestyle='-', linewidth=2.0, alpha=0.85, label='HMS+LSTM')

        ax.set_ylim(bottom=0)
        ax.set_ylabel('Streamflow (CFS)')
        ax.annotate(BASIN_LABELS[basin], xy=(-0.08, 0.5), xycoords='axes fraction', fontsize=13, fontweight='bold', rotation=90, va='center', ha='center')
        ax.tick_params(axis='x', rotation=30)
        ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%m/%d'))
        ax.grid(alpha=0.15)
        if basin == BASINS[0]:
            ax.legend(loc='upper right', framealpha=0.9)
            ax.set_title(f'Extreme Holdout - Largest Peak Event ({scale_label})', fontweight='bold')

    plt.tight_layout()
    save_path = SAVE_DIR / f'sq2_blend_hydrograph_{scale_label.lower()}.png'
    fig.savefig(save_path)
    plt.show()
    print(f'Saved: {save_path}')

print('\n--- Blend NSE Comparison (Extreme) ---')
for basin in BASINS:
    for scale in ['1D', '1H']:
        df = align_all_models(basin, scale, 'EXTREME')
        if df is None: continue
        clean = df.dropna(subset=['Observed'])
        obs = clean['Observed'].values
        for model in ['HMS', 'LSTM', 'PILSTM']:
            if model in clean.columns:
                pred = clean[model].dropna().values
                if len(pred) == len(obs):
                    nse = 1 - np.sum((obs - pred)**2) / np.sum((obs - np.mean(obs))**2)
        if 'HMS' in clean.columns and 'LSTM' in clean.columns:
            hms_v = clean['HMS'].values
            lstm_v = clean['LSTM'].values
            blend_v = (hms_v + lstm_v) / 2.0
            mask = ~(np.isnan(hms_v) | np.isnan(lstm_v))
            blend_nse = 1 - np.sum((obs[mask] - blend_v[mask])**2) / np.sum((obs[mask] - np.mean(obs[mask]))**2)
            hms_nse = 1 - np.sum((obs[mask] - hms_v[mask])**2) / np.sum((obs[mask] - np.mean(obs[mask]))**2)
            lstm_nse = 1 - np.sum((obs[mask] - lstm_v[mask])**2) / np.sum((obs[mask] - np.mean(obs[mask]))**2)
            winner = 'BLEND' if blend_nse > max(hms_nse, lstm_nse) else 'individual'
            print(f'  {BASIN_LABELS[basin]} {scale}: HMS={hms_nse:.4f} LSTM={lstm_nse:.4f} Blend={blend_nse:.4f} -> {winner}')

## SQ3 - BC/Flow Ablation

In [ ]:
abl_rows = []
for exp_key, exp_label in [('CV5', 'Baseline'), ('NOBC', 'NOBC'), ('NOBC_V2', 'NOBC_V2')]:
    for basin in NOBC_BASINS:
        if basin not in data[exp_key]: continue
        for scale in ['1D', '1H']:
            freq_label = 'Daily' if scale == '1D' else 'Hourly'
            for model_key, model_label in [('lstm', 'LSTM'), ('pilstm', 'PILSTM')]:
                ml_df = data[exp_key][basin][model_key][scale]
                if ml_df is None: continue
                clean = ml_df.dropna(subset=['Observed', 'Predicted'])
                obs, pred = clean['Observed'].values, clean['Predicted'].values
                nse, kge = compute_nse_kge(obs, pred)
                abl_rows.append({'Experiment': exp_label, 'Basin': BASIN_LABELS[basin], 'Freq': freq_label, 'Model': model_label, 'NSE': nse, 'KGE': kge})

abl_df = pd.DataFrame(abl_rows)

# ---- Figure 1: Ablation NSE delta (grouped by model, bars = NOBC-Baseline and V2-Baseline) ----
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
bar_width = 0.3
ablation_colors = {'NOBC': '#FF7043', 'NOBC_V2': '#AB47BC'}  # orange-red, purple

for row_idx, freq in enumerate(['Daily', 'Hourly']):
    for col_idx, model in enumerate(['LSTM', 'PILSTM']):
        ax = axes[row_idx, col_idx]
        base = abl_df[(abl_df['Freq'] == freq) & (abl_df['Model'] == model) & (abl_df['Experiment'] == 'Baseline')].set_index('Basin')

        x = np.arange(len(NOBC_BASINS))
        basin_labels = [BASIN_LABELS[b] for b in NOBC_BASINS]

        for i, (abl_exp, abl_label, color) in enumerate([('NOBC', 'NOBC', ablation_colors['NOBC']), ('NOBC_V2', 'NOBC_V2', ablation_colors['NOBC_V2'])]):
            abl = abl_df[(abl_df['Freq'] == freq) & (abl_df['Model'] == model) & (abl_df['Experiment'] == abl_label)].set_index('Basin')
            basins_common = base.index.intersection(abl.index)
            delta_nse = abl.loc[basins_common, 'NSE'].values - base.loc[basins_common, 'NSE'].values

            bars = ax.bar(x + i * bar_width - bar_width / 2, delta_nse, bar_width * 0.85, color=color, edgecolor='black', linewidth=0.3, alpha=0.9, label=abl_label if (row_idx == 0 and col_idx == 0) else None)
            for xi, d in zip(x + i * bar_width - bar_width / 2, delta_nse):
                offset = 0.003 if d >= 0 else -0.003
                ax.text(xi, d + offset, f'{d:+.3f}', ha='center', va='bottom' if d >= 0 else 'top', fontsize=9, fontweight='bold')

        ax.axhline(y=0, color='black', linewidth=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(basin_labels, fontweight='bold')
        ax.set_ylabel('Delta NSE')
        ax.set_title(f'{model} - {freq}', fontweight='bold')
        ax.grid(axis='y', alpha=0.1)
        # Symmetric y-axis
        ymax = max(abs(ax.get_ylim()[0]), abs(ax.get_ylim()[1])) + 0.02
        ax.set_ylim(-ymax, ymax)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.02), frameon=True, edgecolor='gray')
fig.suptitle('Ablation NSE Change Relative to Baseline (CV5)', fontweight='bold', y=1.01)
plt.tight_layout(rect=[0, 0.04, 1, 1])
save_path = SAVE_DIR / 'sq3_ablation_delta_nse.png'
fig.savefig(save_path)
plt.show()

print(f'Saved: {save_path}')
print('\n--- Ablation Summary (NSE) ---')
pivot = abl_df.pivot_table(values='NSE', index=['Basin', 'Freq'], columns=['Model', 'Experiment'], aggfunc='first')
print(pivot.to_string(float_format='%.4f'))
print('\n--- Notable KGE changes ---')

for basin in NOBC_BASINS:
    for freq in ['Daily', 'Hourly']:
        base_kge = abl_df[(abl_df['Basin']==BASIN_LABELS[basin]) & (abl_df['Freq']==freq) & (abl_df['Model']=='LSTM') & (abl_df['Experiment']=='Baseline')]['KGE'].values
        nobc_kge = abl_df[(abl_df['Basin']==BASIN_LABELS[basin]) & (abl_df['Freq']==freq) & (abl_df['Model']=='LSTM') & (abl_df['Experiment']=='NOBC')]['KGE'].values
        if len(base_kge) > 0 and len(nobc_kge) > 0:
            d = nobc_kge[0] - base_kge[0]
            if abs(d) > 0.03:
                print(f'  {BASIN_LABELS[basin]} {freq} LSTM KGE: {base_kge[0]:.3f} -> {nobc_kge[0]:.3f} ({d:+.3f})')

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

SCATTER_AXIS_PERCENTILE = 99

for scale, scale_label in [('1D', 'Daily'), ('1H', 'Hourly')]:
    for basin in NOBC_BASINS:
        # Load both baseline and NOBC predictions
        cv5_lstm = data['CV5'][basin]['lstm'][scale]
        nobc_lstm = data['NOBC'][basin]['lstm'][scale]
        if cv5_lstm is None or nobc_lstm is None: continue

        cv5_clean = cv5_lstm.dropna(subset=['Observed', 'Predicted'])
        nobc_clean = nobc_lstm.dropna(subset=['Observed', 'Predicted'])

        # Align on same dates
        merged = cv5_clean[['Date', 'Observed', 'Predicted']].rename(columns={'Predicted': 'CV5'}).merge(
            nobc_clean[['Date', 'Predicted']].rename(columns={'Predicted': 'NOBC'}), on='Date', how='inner')

        obs = merged['Observed'].values
        obs_valid = obs[~np.isnan(obs)]
        axis_cap = np.percentile(obs_valid, SCATTER_AXIS_PERCENTILE) * 1.1
        lims = [0, axis_cap]

        fig, ax = plt.subplots(1, 1, figsize=(7, 7))

        # Plot NOBC first (behind), then CV5 on top
        ax.scatter(merged['Observed'], merged['NOBC'], s=8, alpha=0.2, color='#E53935', label='NOBC (no BCs)', rasterized=True, zorder=2)
        ax.scatter(merged['Observed'], merged['CV5'], s=8, alpha=0.2, color=COLORS['LSTM'], label='Baseline (with BCs)', rasterized=True, zorder=3)

        ax.plot(lims, lims, 'k-', linewidth=0.8, alpha=0.5, label='1:1')
        ax.set_xlim(lims)
        ax.set_ylim(lims)
        ax.set_xlabel('Observed Streamflow (CFS)')
        ax.set_ylabel('Predicted Streamflow (CFS)')
        ax.set_title(f'{BASIN_LABELS[basin]} LSTM - {scale_label}\nBaseline vs NOBC', fontweight='bold')
        ax.set_aspect('equal')
        ax.grid(alpha=0.1)
        ax.legend(markerscale=3, loc='upper left')

        # Compute NSE for both
        mask = ~(np.isnan(merged['Observed']) | np.isnan(merged['CV5']) | np.isnan(merged['NOBC']))
        obs_m = merged['Observed'].values[mask]
        cv5_nse = 1 - np.sum((obs_m - merged['CV5'].values[mask])**2) / np.sum((obs_m - np.mean(obs_m))**2)
        nobc_nse = 1 - np.sum((obs_m - merged['NOBC'].values[mask])**2) / np.sum((obs_m - np.mean(obs_m))**2)
        ax.text(0.03, 0.85, f'Baseline NSE: {cv5_nse:.4f}\nNOBC NSE: {nobc_nse:.4f}\nDelta: {nobc_nse-cv5_nse:+.4f}', transform=ax.transAxes, ha='left', va='top', fontsize=10, bbox=dict(boxstyle='round', facecolor='white', edgecolor='gray', alpha=0.95, pad=0.5))

        # Inset: top 5%
        p95 = np.percentile(obs_valid, 95)
        ax_ins = inset_axes(ax, width="40%", height="40%", loc='lower right', borderpad=2.5)
        high = obs >= p95
        ax_ins.scatter(merged['Observed'][high], merged['NOBC'][high], s=16, alpha=0.5, color='#E53935', edgecolor='white', linewidth=0.3, rasterized=True)
        ax_ins.scatter(merged['Observed'][high], merged['CV5'][high], s=16, alpha=0.5, color=COLORS['LSTM'], edgecolor='white', linewidth=0.3, rasterized=True)
        ins_max = obs[high].max() * 1.1
        ins_min = p95 * 0.8
        ax_ins.plot([ins_min, ins_max], [ins_min, ins_max], 'k-', linewidth=0.6, alpha=0.5)
        ax_ins.set_xlim(ins_min, ins_max)
        ax_ins.set_ylim(ins_min, ins_max)
        ax_ins.set_aspect('equal')
        ax_ins.tick_params(labelsize=8)
        ax_ins.locator_params(nbins=4)
        ax_ins.set_title('Top 5%', fontsize=9, fontweight='bold')
        ax_ins.patch.set_facecolor('white')
        ax_ins.patch.set_alpha(0.95)
        for spine in ax_ins.spines.values():
            spine.set_edgecolor('gray')
            spine.set_linewidth(1.0)

        plt.tight_layout()
        basin_tag = basin.replace('_', '')
        save_path = SAVE_DIR / f'sq3_scatter_overlay_{basin_tag}_{scale_label.lower()}.png'
        fig.savefig(save_path)
        plt.show()
        print(f'Saved: {save_path}')

In [ ]:
fig, axes = plt.subplots(1, len(NOBC_BASINS), figsize=(22, 7))

for ax, basin in zip(axes, NOBC_BASINS):
    # IG data (CV5 LSTM daily)
    ig_key = (basin, 'LSTM', '1D')
    if ig_key not in ig_results:
        ax.text(0.5, 0.5, 'No IG data', ha='center', va='center', transform=ax.transAxes)
        continue

    ig_df = ig_results[ig_key].head(15).copy()
    ig_df['abbrev'] = ig_df['feature'].apply(abbrev_feature)
    ig_df['is_flow'] = ig_df['feature'].str.contains('FLOW', case=False)

    # Bar chart: IG importance, colored by whether it's a FLOW feature
    colors_ig = ['#E53935' if f else '#78909C' for f in ig_df['is_flow']]
    y_pos = np.arange(len(ig_df))

    ax.barh(y_pos, ig_df['mean_abs_attr'].values, color=colors_ig, edgecolor='black', linewidth=0.3, alpha=0.85)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(ig_df['abbrev'].values, fontsize=11)
    ax.invert_yaxis()
    ax.set_xlabel('Mean |Attribution|')
    ax.set_title(f'{BASIN_LABELS[basin]}', fontweight='bold')

    # Annotate NOBC delta
    cv5_nse_val = abl_df[(abl_df['Basin']==BASIN_LABELS[basin]) & (abl_df['Freq']=='Daily') & (abl_df['Model']=='LSTM') & (abl_df['Experiment']=='Baseline')]['NSE'].values
    nobc_nse_val = abl_df[(abl_df['Basin']==BASIN_LABELS[basin]) & (abl_df['Freq']=='Daily') & (abl_df['Model']=='LSTM') & (abl_df['Experiment']=='NOBC')]['NSE'].values
    if len(cv5_nse_val) > 0 and len(nobc_nse_val) > 0:
        delta = nobc_nse_val[0] - cv5_nse_val[0]
        ax.text(0.95, 0.95, f'NOBC NSE delta:\n{delta:+.3f}', transform=ax.transAxes, ha='right', va='top', fontsize=13, fontweight='bold', color='#D32F2F', bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='#D32F2F', alpha=0.95, pad=0.5))

    # Flow feature count
    n_flow = ig_df['is_flow'].sum()
    ax.text(0.95, 0.75, f'{n_flow} FLOW features\nin top 15', transform=ax.transAxes, ha='right', va='top', fontsize=9, color='#E53935', fontstyle='italic')

from matplotlib.lines import Line2D
legend_elements = [Patch(facecolor='#E53935', edgecolor='black', linewidth=0.3, alpha=0.85, label='FLOW features'),
                   Patch(facecolor='#78909C', edgecolor='black', linewidth=0.3, alpha=0.85, label='Non-FLOW features')]
fig.legend(handles=legend_elements, loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.04), frameon=True, edgecolor='gray')

fig.suptitle('Attribution vs Necessity: IG Importance (LSTM Daily) with Ablation NSE Delta', fontweight='bold', y=1.02)
plt.tight_layout(rect=[0, 0.04, 1, 1])
save_path = SAVE_DIR / 'sq3_attribution_vs_necessity.png'
fig.savefig(save_path)
plt.show()
print(f'Saved: {save_path}')
print('\n--- Attribution != Necessity ---')

for basin in NOBC_BASINS:
    ig_key = (basin, 'LSTM', '1D')
    if ig_key not in ig_results: continue
    ig_df = ig_results[ig_key]
    flow_feats = ig_df[ig_df['feature'].str.contains('FLOW', case=False)]
    if not flow_feats.empty:
        top_flow_rank = int(flow_feats.iloc[0]['rank'])
        total = len(ig_df)
        cv5_v = abl_df[(abl_df['Basin']==BASIN_LABELS[basin]) & (abl_df['Freq']=='Daily') & (abl_df['Model']=='LSTM') & (abl_df['Experiment']=='Baseline')]['NSE'].values[0]
        nobc_v = abl_df[(abl_df['Basin']==BASIN_LABELS[basin]) & (abl_df['Freq']=='Daily') & (abl_df['Model']=='LSTM') & (abl_df['Experiment']=='NOBC')]['NSE'].values[0]
        print(f'  {BASIN_LABELS[basin]}: Top FLOW at IG rank {top_flow_rank}/{total}, but NOBC delta = {nobc_v-cv5_v:+.3f}')

In [ ]:
WINDOW = pd.Timedelta(days=12)

for scale, scale_label in [('1D', 'Daily'), ('1H', 'Hourly')]:
    fig, axes = plt.subplots(len(NOBC_BASINS), 1, figsize=(14, 10))

    for ax, basin in zip(axes, NOBC_BASINS):
        # Get the biggest peak from CV5
        df_cv5 = align_all_models(basin, scale, 'CV5')
        df_nobc = align_all_models(basin, scale, 'NOBC')
        if df_cv5 is None or df_nobc is None: continue

        peak_dates, peak_vals = detect_peaks(df_cv5['Observed'], n_peaks=1, distance=30)
        if len(peak_dates) == 0: continue
        event = peak_dates[0]

        wdf_cv5 = df_cv5.loc[event - WINDOW : event + WINDOW]
        wdf_nobc = df_nobc.loc[event - WINDOW : event + WINDOW]

        # Observed (same for both)
        ax.plot(wdf_cv5.index, wdf_cv5['Observed'], color='black', linewidth=2.5, alpha=0.9, label='Observed')

        # Baseline LSTM
        ax.plot(wdf_cv5.index, wdf_cv5['LSTM'], color=COLORS['LSTM'], linestyle='-', linewidth=1.8, alpha=0.85, label='LSTM (with BCs)')

        # NOBC LSTM
        ax.plot(wdf_nobc.index, wdf_nobc['LSTM'], color='#C62828', linestyle=(0, (5, 3)), linewidth=2.0, alpha=0.9, label='LSTM (no BCs)')

        # HMS for reference
        if 'HMS' in wdf_cv5.columns:
            ax.plot(wdf_cv5.index, wdf_cv5['HMS'], color=COLORS['HMS'], linestyle=':', linewidth=1.5, alpha=0.65, label='HMS')

        ax.set_ylim(bottom=0)
        ax.set_ylabel('Streamflow (CFS)')
        ax.annotate(BASIN_LABELS[basin], xy=(-0.08, 0.5), xycoords='axes fraction', fontsize=13, fontweight='bold', rotation=90, va='center', ha='center')
        ax.tick_params(axis='x', rotation=30)
        ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%m/%d'))
        ax.grid(alpha=0.15)

        # NSE annotation
        obs_clean = wdf_cv5['Observed'].dropna()
        cv5_pred = wdf_cv5.loc[obs_clean.index, 'LSTM'].dropna()
        nobc_pred = wdf_nobc.loc[obs_clean.index, 'LSTM'].dropna()
        common = obs_clean.index.intersection(cv5_pred.index).intersection(nobc_pred.index)
        if len(common) > 10:
            o = obs_clean.loc[common].values
            cv5_r2 = 1 - np.sum((o - cv5_pred.loc[common].values)**2) / np.sum((o - np.mean(o))**2)
            nobc_r2 = 1 - np.sum((o - nobc_pred.loc[common].values)**2) / np.sum((o - np.mean(o))**2)

        if basin == NOBC_BASINS[0]:
            ax.legend(loc='upper right', framealpha=0.9)
            ax.set_title(f'Ablation Hydrograph Overlay - Largest Peak ({scale_label})', fontweight='bold')

    plt.tight_layout()
    save_path = SAVE_DIR / f'sq3_nobc_hydrograph_overlay_{scale_label.lower()}.png'
    fig.savefig(save_path)
    plt.show()
    print(f'Saved: {save_path}')

### SQ3 - IG Attribution: NOBC + NOBC_V2 (compute + compare)

In [ ]:
from UCB_training.ucb_captum import run_ig_analysis_mts

N_SAMPLES = 50
N_STEPS = 50

def find_member_run_dirs(basin, exp_key, model_prefix, n_members=10):
    if exp_key not in run_dirs or basin not in run_dirs[exp_key]:
        return []
    exp_dir = run_dirs[exp_key][basin]
    dirs = []
    for m in range(1, n_members + 1):
        member_dir = exp_dir / f'{model_prefix}_member_run_{m:02d}'
        if not member_dir.exists(): continue
        test_runs = sorted(member_dir.glob('testing_run_*'))
        if test_runs:
            dirs.append(test_runs[-1])
    return dirs

# Run IG for NOBC and NOBC_V2
for exp_key, exp_label in [('NOBC', 'NOBC'), ('NOBC_V2', 'NOBC_V2')]:
    print(f'\n{"="*60}')
    print(f'IG Attribution: {exp_label}')
    print(f'{"="*60}')

    for basin in NOBC_BASINS:
        for model_prefix, model_label in [('nophys', 'LSTM'), ('phys', 'PILSTM')]:
            member_dirs = find_member_run_dirs(basin, exp_key, model_prefix)
            if not member_dirs:
                print(f'  {BASIN_LABELS[basin]} {model_label}: no members found, skipping')
                continue
            print(f'\n  {BASIN_LABELS[basin]} {model_label}: {len(member_dirs)} members')

            for freq_target, freq_label in [('1D', 'Daily'), ('1H', 'Hourly')]:
                all_dfs = []
                for idx, rd in enumerate(member_dirs):
                    try:
                        df_ig = run_ig_analysis_mts(rd, target_freq=freq_target, vary_freq=freq_target, n_samples=N_SAMPLES, n_steps=N_STEPS, data_dir=str(DATA_DIR))
                        all_dfs.append(df_ig)
                        print(f'    Member {idx+1}/{len(member_dirs)} {freq_label} done')
                    except Exception as e:
                        print(f'    Member {idx+1}/{len(member_dirs)} {freq_label} FAILED: {e}')

                if all_dfs:
                    avg_df = all_dfs[0].copy()
                    avg_df['mean_abs_attr'] = np.mean([d['mean_abs_attr'].values for d in all_dfs], axis=0)
                    avg_df['rank'] = avg_df['mean_abs_attr'].rank(ascending=False).astype(int)
                    avg_df = avg_df.sort_values('mean_abs_attr', ascending=False).reset_index(drop=True)

                    # Save to ig_results dict and CSV
                    ig_results[(basin, model_label, freq_target, exp_label)] = avg_df
                    basin_tag = basin.replace('_', '')
                    csv_path = SAVE_DIR / f'sq3_ig_{basin_tag}_{model_label.lower()}_{freq_target.lower()}_{exp_label.lower()}.csv'
                    avg_df.to_csv(csv_path, index=False)

                    print(f'    {BASIN_LABELS[basin]} {model_label} {freq_label} top 3:')
                    for _, row in avg_df.head(3).iterrows():
                        print(f'      {abbrev_feature(row["feature"])} ({row["mean_abs_attr"]:.6f})')

print(f'\nSaved all NOBC/V2 IG CSVs to {SAVE_DIR}')

In [ ]:
for freq, freq_label in [('1D', 'Daily'), ('1H', 'Hourly')]:
    for model_label in ['LSTM', 'PILSTM']:
        fig, axes = plt.subplots(1, len(NOBC_BASINS), figsize=(22, 8))

        for ax, basin in zip(axes, NOBC_BASINS):
            # Load all 3 experiment IGs
            exps_data = {}
            for exp_label in ['CV5', 'NOBC', 'NOBC_V2']:
                # CV5 uses 3-tuple key, NOBC/V2 uses 4-tuple
                if exp_label == 'CV5':
                    key = (basin, model_label, freq)
                    if key in ig_results:
                        exps_data[exp_label] = ig_results[key]
                else:
                    key = (basin, model_label, freq, exp_label)
                    if key in ig_results:
                        exps_data[exp_label] = ig_results[key]

            if len(exps_data) < 2:
                ax.text(0.5, 0.5, f'Missing IG data\n({len(exps_data)}/3 experiments)', ha='center', va='center', transform=ax.transAxes)
                continue

            # Get union of top 10 features from each experiment
            top_features = set()
            for exp_df in exps_data.values():
                top_features.update(exp_df.head(10)['feature'].tolist())
            top_features = sorted(top_features)

            # Build comparison matrix: feature x experiment
            exp_colors = {'CV5': COLORS['LSTM'], 'NOBC': '#FF7043', 'NOBC_V2': '#AB47BC'}
            bar_w = 0.25
            y_pos = np.arange(len(top_features))

            for i, (exp_label, exp_df) in enumerate(exps_data.items()):
                feat_map = dict(zip(exp_df['feature'], exp_df['mean_abs_attr']))
                vals = [feat_map.get(f, 0.0) for f in top_features]
                ax.barh(y_pos + i * bar_w, vals, bar_w * 0.9, color=exp_colors[exp_label], edgecolor='black', linewidth=0.2, alpha=0.85, label=exp_label if basin == NOBC_BASINS[0] else None)

            ax.set_yticks(y_pos + bar_w)
            ax.set_yticklabels([abbrev_feature(f) for f in top_features], fontsize=9)
            ax.invert_yaxis()
            ax.set_xlabel('Mean |Attribution|')
            ax.set_title(BASIN_LABELS[basin], fontweight='bold')

            # Highlight FLOW features
            for j, f in enumerate(top_features):
                if 'FLOW' in f.upper():
                    ax.get_yticklabels()[j].set_color('#E53935')
                    ax.get_yticklabels()[j].set_fontweight('bold')

        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.03), frameon=True, edgecolor='gray')
        fig.suptitle(f'{model_label} {freq_label} - IG Rank Shift (CV5 vs NOBC vs NOBC_V2)', fontweight='bold', y=1.01)
        plt.tight_layout(rect=[0, 0.04, 1, 1])
        save_path = SAVE_DIR / f'sq3_ig_rank_shift_{model_label.lower()}_{freq_label.lower()}.png'
        fig.savefig(save_path)
        plt.show()
        print(f'Saved: {save_path}')

        # Print rank shifts for FLOW features
        print(f'\n--- {model_label} {freq_label} FLOW Feature Rank Shifts ---')
        for basin in NOBC_BASINS:
            for exp_label in ['CV5', 'NOBC', 'NOBC_V2']:
                key = (basin, model_label, freq) if exp_label == 'CV5' else (basin, model_label, freq, exp_label)
                if key not in ig_results: continue
                df = ig_results[key]
                flow_feats = df[df['feature'].str.contains('FLOW', case=False)]
                if not flow_feats.empty:
                    top = flow_feats.iloc[0]
                    print(f'  {BASIN_LABELS[basin]} {exp_label}: {abbrev_feature(top["feature"])} rank={int(top["rank"])}/{len(df)}')